In [1]:
import cantera as ct
import numpy as np

# --- 1. Define Inputs and Initial State ---
# Rocket Parameters
P_chamber_psia = 300.0
P_exit_psia = 10.0
# P_ambient_psia = P_exit_psia # Assuming optimum expansion for initial Isp calculation
P_ambient_psia = 14.7 # Standard ambient pressure at sea level for reference

# Conversion factors
PSI_TO_PA = 6894.76
P_chamber = P_chamber_psia * PSI_TO_PA
P_exit = P_exit_psia * PSI_TO_PA
P_ambient = P_ambient_psia * PSI_TO_PA
g0 = 9.80665 # Standard gravity (m/s^2)

# Propellant Temperatures
T_LOX_in = 90.170 # K
T_RP1_in = 298.15 # K

# Reactant composition based on user's previous input: 1.74 mol C12H26 and 18.5 mol O2
n_fuel_in = 1.74
n_oxidizer_in = 18.5

# --- 2. Setup Cantera Phase and Initial Mixture ---
try:
    # Load the custom gas phase definition
    gas = ct.Solution('rp1_lox_gas.yaml', 'rp1_lox_gas')
except:
    print("FATAL ERROR: Could not load 'rp1_lox_gas.yaml'.")
    print("Please ensure the YAML content provided above is saved as 'rp1_lox_gas.yaml' in the same directory.")
    exit()

# Get Molar Weights
MW_C12H26 = gas.molecular_weights[gas.species_index('C12H26')]
MW_O2 = gas.molecular_weights[gas.species_index('O2')]

# Calculate initial mass of propellants
m_fuel_in = n_fuel_in * MW_C12H26
m_oxidizer_in = n_oxidizer_in * MW_O2
m_total_in = m_fuel_in + m_oxidizer_in

# Calculate initial Mass Fractions (Y)
Y_in = np.zeros(gas.n_species)
Y_in[gas.species_index('C12H26')] = m_fuel_in / m_total_in
Y_in[gas.species_index('O2')] = m_oxidizer_in / m_total_in

# --- 3. Chamber Equilibrium Calculation (Adiabatic Flame Temperature) ---

# Set initial state of the mixture
gas.set_mass_fractions(Y_in)
gas.P = P_chamber

# Calculate average specific enthalpy (h) of the reactants at their inlet temperatures
# H_in = sum(n_i * h_i(T_i)) / m_total
h_c_in = (n_fuel_in * gas.species_enthalpies_mole[gas.species_index('C12H26'), T_RP1_in] + 
          n_oxidizer_in * gas.species_enthalpies_mole[gas.species_index('O2'), T_LOX_in]) / m_total_in

# Set the gas to the initial enthalpy and pressure
gas.HP = h_c_in, P_chamber

# Find Chamber Equilibrium (Adiabatic Flame Temperature)
gas.equilibrate('HP', solver='gibbs', max_steps=1000) 

# Save Chamber Properties
T_chamber = gas.T
MW_chamber = gas.mean_molecular_weight
Cp_chamber = gas.cp_mass
Cv_chamber = gas.cv_mass
gamma_chamber = Cp_chamber / Cv_chamber
h_chamber = gas.enthalpy_mass
s_chamber = gas.entropy_mass
X_frozen = gas.X # Composition is frozen from here

# --- 4. Throat State Calculation (Frozen Flow, Isentropic Expansion) ---

# Set composition to the fixed chamber composition (Frozen Flow Assumption)
gas.X = X_frozen 

# Find the throat pressure by iterating to maximize mass flux (rho*u)
def mass_flux_frozen(P_test):
    """Calculates rho*u for isentropic, frozen expansion to P_test."""
    gas.SP = s_chamber, P_test # Isentropic expansion (S=const) with fixed composition (X=const)
    h_test = gas.enthalpy_mass
    rho_test = gas.density
    
    u_test = np.sqrt(2 * (h_chamber - h_test)) if (h_chamber - h_test) >= 0 else 0.0
    return rho_test * u_test, u_test

# Iterate to find the max flux (throat)
P_range = np.linspace(P_chamber * 0.4, P_chamber * 0.7, 100)
flux_values = []
u_values = []

for P in P_range:
    flux, u = mass_flux_frozen(P)
    flux_values.append(flux)
    u_values.append(u)

idx_throat = np.argmax(flux_values)
P_throat = P_range[idx_throat]
mass_flux_throat = flux_values[idx_throat]
u_throat = u_values[idx_throat]

# C* (Characteristic Velocity) Calculation: C* = Pc / (rho*u)_t
C_star_model = P_chamber / mass_flux_throat

# Throat Properties at P_throat
gas.SP = s_chamber, P_throat 
T_throat = gas.T
rho_throat = gas.density
a_throat = gas.sound_speed # Frozen speed of sound at throat

# --- 5. Exit State and Performance Calculation (Frozen Flow) ---

# Isentropic expansion to Exit Pressure
gas.SP = s_chamber, P_exit 
T_exit = gas.T
h_exit = gas.enthalpy_mass
rho_exit = gas.density
a_exit = gas.sound_speed # Frozen speed of sound at exit

# Exit Velocity (v_e): $v_e = \sqrt{2 \cdot (h_c - h_e)}$
v_exit = np.sqrt(2 * (h_chamber - h_exit))

# Exhaust Mach Number (M_e): $M_e = v_e / a_e$
M_exit = v_exit / a_exit

# Area Ratio ($A_e/A_t$): $\frac{A_e}{A_t} = \frac{\rho_t u_t}{\rho_e v_e}$
A_ratio = mass_flux_throat / (rho_exit * v_exit)

# Thrust Coefficient ($C_f$) - Assuming $\text{P}_e \ne \text{P}_{amb}$
# $C_f = \frac{v_e}{C^*} + \frac{(P_e - P_{amb}) A_e}{P_c A_t}$
C_f_model = (v_exit / C_star_model) + ((P_exit - P_ambient) * A_ratio / P_chamber)

# Specific Impulse ($I_{sp}$) - Total impulse per unit weight flow (using $g_0$)
# $I_{sp} = \frac{F}{\dot{m} g_0} = \frac{C_f C^*}{g_0}$
I_sp_model = (C_f_model * C_star_model) / g0

# --- 6. Results Output ---

print(f"Propellant Ratio: {n_fuel_in} mol C12H26 / {n_oxidizer_in} mol O2")
print(f"Chamber Pressure: {P_chamber_psia} psia ({P_chamber/1e5:.2f} bar)")
print(f"Exit Pressure: {P_exit_psia} psia ({P_exit/1e5:.2f} bar)")
print("\n--- Rocket Performance Parameters (Frozen Flow Model) ---")

# Required output table
results = {
    "Adiabatic Flame Temperature (K)": T_chamber,
    "Chamber Molecular Weight (kg/kmol)": MW_chamber,
    "Chamber Specific Heat Ratio ($\gamma$)": gamma_chamber,
    "Exhaust Mach Number ($M_e$)": M_exit,
    "Exhaust Temperature (K)": T_exit,
    "Exhaust Velocity ($v_e$) (m/s)": v_exit,
    "Specific Impulse ($I_{sp}$) (s)": I_sp_model
}

# Print the table requested in the image
print("\n| Quantity | Model Value |")
print("|:---|:---:|")
for key, value in results.items():
    print(f"| {key} | {value:.3f} |")

# Additional parameters for reference
print(f"\nCharacteristic Velocity ($C^*$) (m/s): {C_star_model:.2f}")
print(f"Thrust Coefficient ($C_f$): {C_f_model:.4f}")
print(f"Area Ratio ($A_e/A_t$): {A_ratio:.2f}")

<>:152: SyntaxWarning: invalid escape sequence '\g'
<>:152: SyntaxWarning: invalid escape sequence '\g'
/var/folders/7s/1ymn0wr17_1gydg004xh37180000gn/T/ipykernel_57321/2237037655.py:152: SyntaxWarning: invalid escape sequence '\g'
  "Chamber Specific Heat Ratio ($\gamma$)": gamma_chamber,


FATAL ERROR: Could not load 'rp1_lox_gas.yaml'.
Please ensure the YAML content provided above is saved as 'rp1_lox_gas.yaml' in the same directory.


/var/folders/7s/1ymn0wr17_1gydg004xh37180000gn/T/ipykernel_57321/2237037655.py:152: SyntaxWarning: invalid escape sequence '\g'
  "Chamber Specific Heat Ratio ($\gamma$)": gamma_chamber,


NameError: name 'gas' is not defined

: 